<a href="https://colab.research.google.com/github/Michele-Maestrini/FusionCore/blob/main/Version%20V0/Notebooks/04b_SOTA_DeepAR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FusionCore v0 — Phase 4b: DeepAR Disqualification Record

**Notebook:** `04b_SOTA_DeepAR.ipynb`  
**Phase:** 4 of 5 (Part B)  
**Objective:** Document and programmatically verify DeepAR's incompatibility with
the FusionCore zero-leakage protocol. **No model is trained. No predictions are produced.**

---

### Disqualification Summary

NeuralForecast's DeepAR implementation has **two independent architectural barriers**
that make it fundamentally incompatible with leakage-free RUL prediction:

| # | Barrier | Error Raised | Impact |
|---|---------|-------------|--------|
| 1 | **No exogenous feature support** | `DeepAR does not support historical exogenous variables.` | Model cannot ingest sensor features — predictions would be based solely on past target values |
| 2 | **Cannot exclude past target** | `DeepAR has no possibility for excluding y.` | Model always sees past RUL values, enabling trivial `RUL[T+1] ≈ RUL[T] − 1` |

**Consequence:** DeepAR is doubly disqualified. Even if barrier #2 could be
circumvented, barrier #1 prevents the model from learning any physics. A model
that predicts RUL from RUL alone — with zero sensor input — is not a prognostics
model.

**Phase 5 operates on three models only: XGBoost, TFT, NHITS.**

---

### References

- **DeepAR:** Salinas, D. et al. (2020). *DeepAR: Probabilistic Forecasting with Autoregressive Recurrent Networks.* IJF.
- **NeuralForecast:** Olivares, K.G. et al. (2022). *NeuralForecast.* Nixtla.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 1 — Environment Setup (Run First)
# ══════════════════════════════════════════════════════════════════════════════

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 2 — Dependency Installation
# ══════════════════════════════════════════════════════════════════════════════

%%capture
!pip install neuralforecast

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 3 — Imports & Constants
# ══════════════════════════════════════════════════════════════════════════════

import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import joblib

DRIVE_ROOT     = Path('/content/drive/MyDrive/PI')
OUTPUTS_DIR    = DRIVE_ROOT / 'FusionCore' / 'v0' / 'outputs'

RUL_CAP      = 125
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)

from neuralforecast.models import DeepAR
from neuralforecast.losses.pytorch import DistributionLoss

nn_feature_names = joblib.load(OUTPUTS_DIR / 'nn_feature_names.pkl')
print(f'NN features: {len(nn_feature_names)}')

print()
print('╔══════════════════════════════════════════════════════════════╗')
print('║  DISQUALIFICATION RECORD                                    ║')
print('║  DeepAR is architecturally incompatible with FusionCore.    ║')
print('║  This notebook verifies both failure modes programmatically.║')
print('║  No model is trained. No predictions are produced.          ║')
print('╚══════════════════════════════════════════════════════════════╝')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 4 — Programmatic Verification: Barrier #1 (No Exogenous Features)
# ══════════════════════════════════════════════════════════════════════════════
# DeepAR in NeuralForecast sets EXOGENOUS_HIST = False. Passing
# hist_exog_list raises an exception before training can begin.

barrier_1_confirmed = False

try:
    model_test = DeepAR(
        h=1,
        input_size=50,
        lstm_hidden_size=64,
        lstm_n_layers=3,
        hist_exog_list=nn_feature_names,
        loss=DistributionLoss(distribution='Normal', level=[80]),
        max_steps=1,
        random_seed=RANDOM_STATE,
    )
    print('ERROR: DeepAR accepted hist_exog_list — investigate.')
except Exception as e:
    barrier_1_confirmed = True
    print('Barrier #1 CONFIRMED — DeepAR rejects exogenous features.')
    print(f'  Exception: {e}')
    print()
    print('  DeepAR.EXOGENOUS_HIST = False')
    print('  The model cannot ingest sensor features (87 NN features).')
    print('  Predictions would be based solely on past target (RUL) values.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 5 — Programmatic Verification: Barrier #2 (Cannot Exclude Past Y)
# ══════════════════════════════════════════════════════════════════════════════
# Even without exogenous features, DeepAR cannot set exclude_insample_y=True.
# The autoregressive decoder requires past y as input by design.

barrier_2_confirmed = False

try:
    model_test = DeepAR(
        h=1,
        input_size=50,
        lstm_hidden_size=64,
        lstm_n_layers=3,
        exclude_insample_y=True,
        loss=DistributionLoss(distribution='Normal', level=[80]),
        max_steps=1,
        random_seed=RANDOM_STATE,
    )
    print('ERROR: DeepAR accepted exclude_insample_y=True — investigate.')
except Exception as e:
    barrier_2_confirmed = True
    print('Barrier #2 CONFIRMED — DeepAR cannot exclude past target values.')
    print(f'  Exception: {e}')
    print()
    print('  The autoregressive decoder feeds y[t-1] as input to predict y[t].')
    print('  For RUL prediction, this enables the trivial shortcut:')
    print('    RUL[T+1] ≈ RUL[T] − 1')
    print('  This constitutes structural target leakage.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 6 — Save Disqualification Record
# ══════════════════════════════════════════════════════════════════════════════

assert barrier_1_confirmed, 'Barrier #1 not confirmed — investigate.'
assert barrier_2_confirmed, 'Barrier #2 not confirmed — investigate.'

joblib.dump({
    'status': 'DISQUALIFIED',
    'reason': (
        'DeepAR (NeuralForecast) has two architectural barriers: '
        '(1) EXOGENOUS_HIST=False — cannot ingest sensor features via hist_exog_list; '
        '(2) exclude_insample_y=True raises Exception — past RUL values are always '
        'exposed as autoregressive input, enabling trivial RUL[T+1] ≈ RUL[T] − 1.'
    ),
    'barrier_1': 'No exogenous feature support (EXOGENOUS_HIST=False)',
    'barrier_2': 'Cannot exclude past target (autoregressive on y)',
    'barrier_1_confirmed': barrier_1_confirmed,
    'barrier_2_confirmed': barrier_2_confirmed,
    'reference_hyperparameters': {
        'h': 1, 'input_size': 50, 'lstm_hidden_size': 64,
        'lstm_n_layers': 3, 'lstm_dropout': 0.2,
    },
}, OUTPUTS_DIR / 'phase4b_deepar_disqualification.pkl')

# Save None as Optuna study — no HPO was performed.
joblib.dump(None, OUTPUTS_DIR / 'optuna_deepar_study.pkl')

print('Phase 4b outputs:')
print(f'  phase4b_deepar_disqualification.pkl  — disqualification record')
print(f'  optuna_deepar_study.pkl              — None (no HPO performed)')
print()
print('NOT produced (by design):')
print(f'  phase4b_deepar_predictions.pkl       — no predictions')
print(f'  deepar_best/                         — no checkpoint')
print()
print('╔══════════════════════════════════════════════════════════════╗')
print('║  ✔ Notebook 04b complete.                                   ║')
print('║  DeepAR disqualified — two architectural barriers confirmed.║')
print('║  Phase 5 evaluates: XGBoost, TFT, NHITS (3 models).        ║')
print('╚══════════════════════════════════════════════════════════════╝')